In [1]:
documents = [
    {
        "id": "gen_2025",
        "text": "Python generators produce values lazily instead of storing all values in memory.",
        "topic": "python",
        "year": 2025
    },
    {
        "id": "gen_2024",
        "text": "Generators produce values one at a time using lazy evaluation.",
        "topic": "python",
        "year": 2024
    },
    {
        "id": "gen_2023",
        "text": "Python generator functions use yield to pause execution and resume later.",
        "topic": "python",
        "year": 2023
    },
    {
        "id": "fastapi_2025",
        "text": "FastAPI is a Python framework used for building web APIs.",
        "topic": "backend",
        "year": 2025
    },
    {
        "id": "redis_2024",
        "text": "Redis is an in-memory data store commonly used for caching and queues.",
        "topic": "backend",
        "year": 2024
    },
    {
        "id": "docker_2025",
        "text": "Docker packages applications and dependencies into containers.",
        "topic": "devops",
        "year": 2025
    }
]

In [2]:
texts = [doc["text"] for doc in documents]
ids = [doc["id"] for doc in documents]

metadatas = [
    {
        "topic": doc["topic"],
        "year": doc["year"]
    }
    for doc in documents
]

Create embeddings

In [4]:
!pip -q install sentence-transformers

In [7]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
embeddings = embedding_model.encode(texts)

Create a fresh collection

In [9]:
!pip -q install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

In [11]:
import chromadb

Client = chromadb.Client()

collection = Client.get_or_create_collection(
    name = "day_10_metadata"
)

Add the documents

In [12]:
collection.add(
    ids = ids,
    documents=texts,
    embeddings = embeddings,
    metadatas = metadatas
)

First: normal semantic search

In [13]:
query = "How do Python generators save memory?"

In [14]:
query_embedding = embedding_model.encode([query])[0]

In [15]:
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

In [16]:
for i in range(len(results["documents"][0])):
    print("Rank:", i + 1)
    print("Document:", results["documents"][0][i])
    print("Metadata:", results["metadatas"][0][i])
    print("Distance:", results["distances"][0][i])
    print()

Rank: 1
Document: Python generators produce values lazily instead of storing all values in memory.
Metadata: {'topic': 'python', 'year': 2025}
Distance: 0.47838354110717773

Rank: 2
Document: Python generator functions use yield to pause execution and resume later.
Metadata: {'year': 2023, 'topic': 'python'}
Distance: 0.689990758895874

Rank: 3
Document: Generators produce values one at a time using lazy evaluation.
Metadata: {'year': 2024, 'topic': 'python'}
Distance: 1.0324528217315674



Add a metadata filter

In [17]:
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3,
    where={
        "year": 2024
    },
    include=["documents", "metadatas", "distances"]
)

In [18]:
for i in range(len(results["documents"][0])):
    print("Rank:", i + 1)
    print("Document:", results["documents"][0][i])
    print("Metadata:", results["metadatas"][0][i])
    print()

Rank: 1
Document: Generators produce values one at a time using lazy evaluation.
Metadata: {'year': 2024, 'topic': 'python'}

Rank: 2
Document: Redis is an in-memory data store commonly used for caching and queues.
Metadata: {'year': 2024, 'topic': 'backend'}



Create an evaluation set

In [19]:
eval_questions = [
    {
        "question": "How do Python generators save memory?",
        "relevant_sources": ["gen_2025", "gen_2024", "gen_2023"],
        "filter": None
    },
    {
        "question": "How do Python generators save memory?",
        "relevant_sources": ["gen_2024"],
        "filter": {"year": 2024}
    },
    {
        "question": "How do Python generators save memory?",
        "relevant_sources": ["gen_2025"],
        "filter": {
            "$and": [
                {"topic": "python"},
                {"year": 2025}
            ]
        }
    },
    {
        "question": "What is FastAPI used for?",
        "relevant_sources": ["fastapi_2025"],
        "filter": {"topic": "backend"}
    }
]

 Write a generic Recall@K evaluator

In [20]:
def recall_at_k(item, k=3):
    question = item["question"]
    relevant_sources = set(item["relevant_sources"])
    metadata_filter = item["filter"]

    query_embedding = embedding_model.encode([question])[0]

    kwargs = {
        "query_embeddings": [query_embedding.tolist()],
        "n_results": k,
        "include": ["documents", "metadatas"]
    }

    if metadata_filter is not None:
        kwargs["where"] = metadata_filter

    results = collection.query(**kwargs)

    retrieved_ids = results["ids"][0]

    return bool(relevant_sources.intersection(retrieved_ids))

In [21]:
for k in [1, 2, 3]:
    successes = 0

    for item in eval_questions:
        if recall_at_k(item, k):
            successes += 1

    recall = successes / len(eval_questions)

    print(f"Recall@{k}: {recall:.2f}")

Recall@1: 1.00
Recall@2: 1.00
Recall@3: 1.00


Let's directly compare filtered vs unfiltered retrieval

In [22]:
def retrieve(question, k=3, metadata_filter=None):

    query_embedding = embedding_model.encode([question])[0]

    kwargs = {
        "query_embeddings": [query_embedding.tolist()],
        "n_results": k,
        "include": ["documents", "metadatas", "distances"]
    }

    if metadata_filter is not None:
        kwargs["where"] = metadata_filter

    return collection.query(**kwargs)

In [23]:
query = "How do Python generators save memory?"

print("===== NO FILTER =====")

results = retrieve(query, k=3)

for i in range(len(results["documents"][0])):
    print(
        i + 1,
        results["ids"][0][i],
        results["metadatas"][0][i]
    )

===== NO FILTER =====
1 gen_2025 {'topic': 'python', 'year': 2025}
2 gen_2023 {'topic': 'python', 'year': 2023}
3 gen_2024 {'topic': 'python', 'year': 2024}


In [24]:
print("===== YEAR = 2024 =====")

results = retrieve(
    query,
    k=3,
    metadata_filter={"year": 2024}
)

for i in range(len(results["documents"][0])):
    print(
        i + 1,
        results["ids"][0][i],
        results["metadatas"][0][i]
    )

===== YEAR = 2024 =====
1 gen_2024 {'topic': 'python', 'year': 2024}
2 redis_2024 {'topic': 'backend', 'year': 2024}


# Retrieval Experiments

## Day 10 — Metadata Filtering

### Experiment 1: Metadata Filtering

#### Goal

Understand whether structured metadata constraints
improve retrieval quality for constrained questions.

#### Baseline

Semantic retrieval without metadata filtering.

#### Experiment

Compared:

1. No metadata filter
2. `year = 2024`
3. `topic = python`
4. `topic = python AND year = 2025`

#### Metrics

- Recall@1
- Recall@2
- Recall@3

#### Results

| Configuration | Recall@1 | Recall@2 | Recall@3 |
|---|---:|---:|---:|
| No filter | TODO | TODO | TODO |
| Year = 2024 | TODO | TODO | TODO |
| Python + 2025 | TODO | TODO | TODO |

#### Observations

- Metadata filtering enforces explicit structured constraints.
- Filtering can prevent semantically similar but invalid documents
  from entering the retrieval results.
- Overly restrictive filters can reduce recall.
- Retrieval quality depends on both semantic relevance and valid metadata constraints.

#### Key lesson

Metadata filtering is not a replacement for semantic search.
It is a complementary retrieval mechanism that allows an AI system
to enforce structured constraints.